In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv")

In [2]:
def episcope_combis(path: str, country: str = "DE", building_types: list = ["SFH", "MFH", "TH", "AB"])-> dict:
    df = pd.read_csv(path, delimiter=",")
    df = df.copy()
    df = df[(df["Code_Country"] == country) & (df["Code_BuildingSizeClass"].isin(building_types)) & (df["Code_DataType_Building"] == "ReEx") & (df["Code_BuildingVariant"].str.contains(".N.", regex=False))]
    cons = {
        "B_N2": "Terraced",
        "B_N1": "Semi",
        "B_Alone": "Detached"
    }
    df["Connections"] = df["Code_AttachedNeighbours"].map(cons)

    test = df[["Code_BuildingSizeClass", "Code_ConstructionYearClass", "Connections"]].drop_duplicates()
    #print(test.index)
    tzu = df.loc[test.index]
    tzu = tzu[["Code_BuildingSizeClass","Connections", "Code_ConstructionYearClass"]]
    tzu.columns = ["Type", "Surrounding","AgeBin"]
    res_dict = tzu.to_dict("records")
    return res_dict
episcope_combis(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv")

[{'Type': 'AB', 'Surrounding': 'Terraced', 'AgeBin': 'DE.02'},
 {'Type': 'AB', 'Surrounding': 'Semi', 'AgeBin': 'DE.03'},
 {'Type': 'AB', 'Surrounding': 'Detached', 'AgeBin': 'DE.04'},
 {'Type': 'AB', 'Surrounding': 'Detached', 'AgeBin': 'DE.05'},
 {'Type': 'AB', 'Surrounding': 'Detached', 'AgeBin': 'DE.06'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.01'},
 {'Type': 'MFH', 'Surrounding': 'Terraced', 'AgeBin': 'DE.02'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.03'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.04'},
 {'Type': 'MFH', 'Surrounding': 'Semi', 'AgeBin': 'DE.05'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.06'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.07'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.08'},
 {'Type': 'MFH', 'Surrounding': 'Semi', 'AgeBin': 'DE.09'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.10'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE

In [354]:
len(episcope_combis(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv"))

40

In [ ]:
#building type
#n apartments
#occ
# flat area



In [336]:
#https://episcope.eu/building-typology/country/de/  Statistics of the German Building Stock Source[2]
def gen_flat_count(buildingtype: str, rng: np.random.Generator) -> int:
    if buildingtype == "SFH":
        return 1
    elif buildingtype == "TH":
        return int(rng.integers(low=1, high=3))
    elif buildingtype == "MFH":
        return int(rng.integers(low=3, high=13)) #exclusive
    elif buildingtype == "AB":
        return int(rng.integers(low=13, high=21))
    else:
        raise ValueError

gen_flat_count("AB", draw)



20

In [52]:
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Wohnen/Tabellen/tabelle-wo2-mietwohnungen.html
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Wohnen/Tabellen/tabelle-wo2-eigentuemerwohnungen.html
#probs sind für haushalte
def gen_occs(buildingtype: str, apartment_count: int, rng: np.random.Generator) ->list:

    occ_probs = {
        "SFH": (0.245114, 0.402323, 0.154148, 0.148869, 0.049623),
        "MFH": (0.490622, 0.307419, 0.101949, 0.074417, 0.024805),
        "AB": (0.490622, 0.307419, 0.101949, 0.074417, 0.024805),
        "TH": (0.236817, 0.400092, 0.157261, 0.154371, 0.051457),
    }
    probs = occ_probs[buildingtype]

    num_occs =[]

    for _ in range(apartment_count):
        draw = rng.random()
        x = 0.0
        for i,v in enumerate(probs, start=1):
            x += v
            if x > draw:
                num_occs.append(i)
                break
        else:
            num_occs.append(len(probs))



    return num_occs
draw = np.random.default_rng(45676)
gen_occs("MFH", 5, draw)

[1, 2, 1, 1, 1]

In [361]:
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Wohnen/Tabellen/tabelle-wo4-wohnflaeche.html
def gen_flat_area(occs: list ,rng: np.random.Generator)-> list:
    area_occ_probs = {
        1: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.09605,0.30524,0.27605,0.13575,0.07434,0.05302, 0.05955]
        },
        2: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.00677,0.08684,0.23555,0.19991,0.15465,0.13864, 0.17764]
        },
        3: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.0,0.03780,0.19027,0.20132,0.15757,0.16097, 0.25207]
        },
        4: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.0,0.01389,0.11820,0.17035,0.14979,0.17815, 0.36962]
        },
    }
    areas = []

    for occ in occs:
        if occ > 4:
            occ = 4

        res = rng.choice(area_occ_probs[occ]["area"], p=area_occ_probs[occ]["probs"])
        floor = rng.uniform(low=res[0],high= res[1])
        areas.append(floor)

    return areas



In [351]:
gen_flat_area([2,9,3], draw)

[74.43406419781391, 85.21254189192607, 64.4646854529817]

In [340]:
import itertools
grid = itertools.product([1,2],[3,4])

In [359]:
for _ in range(10):
    print(_)

0
1
2
3
4
5
6
7
8
9


# Generator

In [376]:

def manifest(scenario_reps,seed_reps,episcope_path , seed=42):
    combinations = episcope_combis(episcope_path)
    year_types = ["average", "hot", "cold"]
    future = [False, True]
    rng = np.random.default_rng(seed)
    rows = []
    scenario_id = 0

    grid  = itertools.product(combinations, year_types, future)
    for x,y,z in grid:
        b_type = x["Type"]

        for _ in range(scenario_reps):
            n_flats = gen_flat_count(buildingtype=b_type, rng=rng)
            occs = gen_occs(buildingtype=b_type,apartment_count=n_flats, rng=rng)
            flat_areas = gen_flat_area(occs=occs,rng=rng)
            total_area = sum(flat_areas)
            clima_region = int(rng.integers(low=1, high=16))

            for _ in range(seed_reps):
                rows.append(dict(
                    country = "DE",
                    buildingType = b_type,
                    surrounding = x["Surrounding"],
                    buildingAgeBin = x["AgeBin"],
                    a_ref = round(total_area,1),
                    n_apartments = n_flats,
                    year_type = y,
                    future = z,
                    climateRegion = clima_region,
                    n_persons = occs,
                    freq = "1min",
                    hasFirePlace = False,
                    cores = 1,
                    scenario_id = scenario_id
                )
                    )
            scenario_id += 1
    manifest = pd.DataFrame(rows)
    manifest["run_id"] = manifest.index.map(lambda i: f"run_{i:06d}")
    manifest["seed"]= 1000000 + manifest.index
    return manifest



manifest = manifest(3,20,r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv", seed=45)


In [382]:
manifest

,country,buildingType,surrounding,buildingAgeBin,a_ref,n_apartments,year_type,future,climateRegion,n_persons,freq,hasFirePlace,cores,scenario_id,run_id,seed
0,DE,AB,Terraced,DE.02,1938.3,20,average,False,9,"[2, 2, 3, 2, 2, 2, 2, 1, 2, 2, 3, 2, 2, 4, 1, ...",1min,False,1,0,run_000000,1000000
1,DE,AB,Terraced,DE.02,1938.3,20,average,False,9,"[2, 2, 3, 2, 2, 2, 2, 1, 2, 2, 3, 2, 2, 4, 1, ...",1min,False,1,0,run_000001,1000001
2,DE,AB,Terraced,DE.02,1938.3,20,average,False,9,"[2, 2, 3, 2, 2, 2, 2, 1, 2, 2, 3, 2, 2, 4, 1, ...",1min,False,1,0,run_000002,1000002
3,DE,AB,Terraced,DE.02,1938.3,20,average,False,9,"[2, 2, 3, 2, 2, 2, 2, 1, 2, 2, 3, 2, 2, 4, 1, ...",1min,False,1,0,run_000003,1000003
4,DE,AB,Terraced,DE.02,1938.3,20,average,False,9,"[2, 2, 3, 2, 2, 2, 2, 1, 2, 2, 3, 2, 2, 4, 1, ...",1min,False,1,0,run_000004,1000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14395,DE,TH,Semi,DE.12,145.3,1,cold,True,10,[2],1min,False,1,719,run_014395,1014395
14396,DE,TH,Semi,DE.12,145.3,1,cold,True,10,[2],1min,False,1,719,run_014396,1014396
14397,DE,TH,Semi,DE.12,145.3,1,cold,True,10,[2],1min,False,1,719,run_014397,1014397
14398,DE,TH,Semi,DE.12,145.3,1,cold,True,10,[2],1min,False,1,719,run_014398,1014398


In [386]:
manifest.to_parquet(r"manifest.parquet")

In [387]:
jsh = pd.read_parquet(r"manifest.parquet")

In [389]:
iue = jsh.loc[1, "n_persons"]

In [393]:
iue[19]

1

In [341]:
list(grid)

[(1, 3), (1, 4), (2, 3), (2, 4)]

In [338]:
gen_flat_area([1,2,5],draw)

[35.962335224845816, 92.6675334253968, 121.41659959689358]

In [151]:
1 - sum(area_occ_probs[4]["probs"])

0.0

In [84]:
test = np.random.default_rng(42)


In [169]:
test.choice(area_occ_probs[1]["area"], p=area_occ_probs[1]["probs"])

array([80, 99])

In [285]:
test.integers(low=1,high=3)

np.int64(2)

In [237]:

test.uniform(low=0,high=1)

0.15428949206754783

In [38]:
episcope_combis(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv")

,Code_BuildingVariant,Code_BuildingSizeClass,Code_AttachedNeighbours,Code_ConstructionYearClass
187,DE.N.TH.02.Gen.ReEx.001.001,TH,B_N2,DE.02
188,DE.N.TH.03.Gen.ReEx.001.001,TH,B_N2,DE.03
189,DE.N.TH.04.Gen.ReEx.001.001,TH,B_N1,DE.04
190,DE.N.TH.05.Gen.ReEx.001.001,TH,B_N2,DE.05
191,DE.N.TH.06.Gen.ReEx.001.001,TH,B_N2,DE.06
192,DE.N.TH.07.Gen.ReEx.001.001,TH,B_N2,DE.07
193,DE.N.TH.08.Gen.ReEx.001.001,TH,B_N1,DE.08
194,DE.N.TH.09.Gen.ReEx.001.001,TH,B_N2,DE.09
195,DE.N.TH.10.Gen.ReEx.001.001,TH,B_N1,DE.10
196,DE.N.TH.11.Gen.ReEx.001.001,TH,B_N1,DE.11


In [44]:
df_new = df[(df["Code_Country"] == "DE") &(df["Code_BuildingVariant"].str.contains("SFH"))]

In [49]:
df_new.loc[175:179]

,Unnamed: 0,Code_BuildingVariant,Date_Entry,Code_StatusDataset,Code_Country,Code_Building,Description_BuildingVariant,Description_BuildingVariant_National,Code_BuildingType,Code_DataType_Building,...,Q_Sol_South,Q_Sol_West,Q_Sol_North,q_sol,q_int,tau,a_H,gamma_h_gn,eta_h_gn,q_h_nd
175,545,DE.N.SFH.02.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.02.Gen.ReEx.001,0,0,DE.N.SFH.02.Gen,ReEx,...,625.51440,481.46805,48.818700,12.128378,15.552,9.720661,1.124022,0.093622,0.936326,269.742045
176,548,DE.N.SFH.03.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.03.Gen.ReEx.001,0,0,DE.N.SFH.03.Gen,ReEx,...,2457.37800,572.72670,423.676575,13.634850,15.552,12.078144,1.202605,0.118778,0.931395,218.542445
177,551,DE.N.SFH.04.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.04.Gen.ReEx.001,0,0,DE.N.SFH.04.Gen,ReEx,...,960.61140,204.54525,115.072650,13.679449,15.552,9.576125,1.119204,0.097398,0.932917,272.851886
178,554,DE.N.SFH.05.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.05.Gen.ReEx.001,0,0,DE.N.SFH.05.Gen,ReEx,...,703.70370,560.13930,142.969050,15.071433,15.552,9.783928,1.126131,0.104251,0.929209,265.292457
179,557,DE.N.SFH.06.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.06.Gen.ReEx.001,0,0,DE.N.SFH.06.Gen,ReEx,...,1858.67136,314.68500,263.969685,16.195533,15.552,13.411644,1.247055,0.141350,0.924217,195.260705


In [45]:
df_new[["Code_BuildingVariant", 'Code_AttachedNeighbours', "Code_StatusDataset", "Code_Country"]]

,Code_BuildingVariant,Code_AttachedNeighbours,Code_StatusDataset,Code_Country
174,DE.N.SFH.01.Gen.ReEx.001.001,B_Alone,Typology,DE
175,DE.N.SFH.02.Gen.ReEx.001.001,B_Alone,Typology,DE
176,DE.N.SFH.03.Gen.ReEx.001.001,B_Alone,Typology,DE
177,DE.N.SFH.04.Gen.ReEx.001.001,B_Alone,Typology,DE
178,DE.N.SFH.05.Gen.ReEx.001.001,B_Alone,Typology,DE
179,DE.N.SFH.06.Gen.ReEx.001.001,B_Alone,Typology,DE
180,DE.N.SFH.06.LightFrame.ReEx.001.001,B_Alone,Typology,DE
181,DE.N.SFH.07.Gen.ReEx.001.001,B_N1,Typology,DE
182,DE.N.SFH.08.Gen.ReEx.001.001,B_N1,Typology,DE
183,DE.N.SFH.09.Gen.ReEx.001.001,B_Alone,Typology,DE
